# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Comparación final

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

Balanced accuracy pondera por igual el reconocimiento de daño y de seguro bajo desbalance [1]. Las curvas precisión–recall siguen siendo salvaguardas informativas [2], pero no se suman a BA con pesos arbitrarios: el aprendizaje multiobjetivo recomienda explicitar preferencias y soluciones Pareto [3]. La calibración se audita [4], la revisión se informa con riesgo–cobertura [5], y test permanece fuera de toda selección para evitar sesgo [6]. La capacidad humana, el margen de no inferioridad y los costos son decisiones locales que deben predeclararse.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


## Restauración reproducible del dataset

In [ ]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


## Configuración y ejecución

In [ ]:
from moderacion_peru.ensemble_evaluation import compare_and_freeze_validation,evaluate_frozen_test
DATA=ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
CANDIDATE_ROOTS=[ROOT/'modelos/v2']
COMPARISON=ROOT/'resultados/modelos/comparacion_individual_ensemble_validation.json'
FREEZE=ROOT/'resultados/modelos/seleccion_congelada.json'
TEST_REPORT=ROOT/'resultados/modelos/test_final_abierto_una_vez.json'
PARALLEL_WORKERS=4  # bootstrap pareado por video
BOOTSTRAP_REPLICATES=2000
SELECTION_FOLDS=5
# Deben acordarse ANTES de comparar. None permite el informe/Pareto, pero mantiene test sellado.
MAX_REVIEW_RATE=None  # p. ej. 0.10 solo si capacidad humana <=10% fue aprobada
MACRO_AUPRC_NONINFERIORITY_MARGIN=None  # p. ej. 0.02 solo si fue predeclarado
RUN_COMPARE_AND_FREEZE=False
RUN_TEST_ONCE=False
RUN_PUBLISH=False
if RUN_COMPARE_AND_FREEZE:
    comparison_result=run_with_progress('BA OOF, riesgo-cobertura y bootstrap',compare_and_freeze_validation,DATA,CANDIDATE_ROOTS,COMPARISON,FREEZE,bootstrap_replicates=BOOTSTRAP_REPLICATES,selection_folds=SELECTION_FOLDS,max_review_rate=MAX_REVIEW_RATE,macro_auprc_noninferiority_margin=MACRO_AUPRC_NONINFERIORITY_MARGIN,parallel_workers=PARALLEL_WORKERS,progress_unit='réplica')
    show_result('Comparación y congelación en validation',comparison_result,tone='success')
if RUN_TEST_ONCE:
    test_result=run_with_progress('Inferencia de test',evaluate_frozen_test,FREEZE,TEST_REPORT,confirm_single_test_open=True,progress_unit='lote')
    show_result('Apertura única de test natural + vista 4:1',test_result,tone='warning')
if RUN_PUBLISH:
    raise RuntimeError('Publicación bloqueada por diseño: habilítela solo tras aprobación posterior y revisión de FREEZE/TEST_REPORT.')
if not (RUN_COMPARE_AND_FREEZE or RUN_TEST_ONCE or RUN_PUBLISH):
    show_summary('Criterio vigente',{'ranking':'BA binaria ANY_DAMAGE OOF a cobertura completa','agregación':'lexicográfica; no suma métricas redundantes','salvaguarda':'macro-AUPRC daños + frontera Pareto','desempate':'menor R_0.67; luego macro-AUPRC','NEEDS_REVIEW':'política posterior bajo capacidad humana declarada','bootstrap':f'{BOOTSTRAP_REPLICATES} réplicas pareadas por video en {PARALLEL_WORKERS} hilos','test':'bloqueado hasta fijar capacidad y margen antes de comparar'},tone='neutral')

## Referencias

[1] K. H. Brodersen, C. S. Ong, K. E. Stephan, et al., "The Balanced Accuracy and Its Posterior Distribution," in Proc. 20th Int. Conf. Pattern Recognition, 2010, pp. 3121–3124, doi: 10.1109/ICPR.2010.764.

[2] T. Saito and M. Rehmsmeier, "The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets," PLOS ONE, vol. 10, no. 3, Art. no. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[3] Y. Jin and B. Sendhoff, "Pareto-Based Multiobjective Machine Learning: An Overview and Case Studies," IEEE Trans. Syst., Man, Cybern. C, vol. 38, no. 3, pp. 397–415, 2008, doi: 10.1109/TSMCC.2008.919172.

[4] C. Guo, G. Pleiss, Y. Sun, et al., "On Calibration of Modern Neural Networks," in Proc. ICML, vol. 70, 2017, pp. 1321–1330. [Online]. Available: https://proceedings.mlr.press/v70/guo17a.html

[5] Y. Geifman and R. El-Yaniv, "Selective Classification for Deep Neural Networks," in Adv. Neural Inf. Process. Syst., vol. 30, 2017.

[6] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.